# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a complete step-by-step guide for loading, exploring, and analyzing a Croissant-structured dataset using the `mlcroissant` library. All `@id` fields are used to refer to dataset entities for clarity and reproducibility.

### Dataset Source
The dataset schema is available at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install `mlcroissant` library
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

meta = dataset.metadata  # Treat metadata as an object

print(f"Dataset: {meta.name}\nDescription: {meta.description}\nPublished: {meta.datePublished}\nLicense: {meta.license}")
print(f"Keywords: {getattr(meta, 'keywords', None)}")

## 2. Data Overview
Let's review the available record sets and their `@id` fields. We also list field `@id`s within those record sets for reference.

Since the Croissant schema can contain multiple record sets, fields, and columns, we print a summary below using only their `@id`s and labels (where available).


In [ ]:
# List all record sets in the dataset
print("Available Record Sets:")
for rs in meta.recordSet:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', None)}")

# Let's select the main record set (by @id)
record_sets = [rs['@id'] for rs in meta.recordSet]

print("\nFields by RecordSet:")
record_set_fields = {}
for rs in meta.recordSet:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name', None)}")
    field_ids = []
    for f in rs.get('field', []):
        if isinstance(f, dict):
            field_id = f['@id']
        else:
            field_id = f
        field_ids.append(field_id)
        # Search for the field in meta.field (if available)
        # Otherwise, just print the ID.
        # Croissant's field definitions can sometimes be under meta.field
        print(f"  - Field @id: {field_id}")
    record_set_fields[rs['@id']] = field_ids

if not record_sets:
    print("No record sets found. Please check the dataset metadata structure.")

## 3. Data Extraction

We use `mlcroissant` to extract records from one or more record sets. Each record set is referenced by its `@id` field. All DataFrame columns use their corresponding Croissant field `@id` as label.

**Example:** The main protein records may be in a record set with an `@id` such as `_:protein_table` (replace with actual ID from previous section). Adjust accordingly if more than one record set is of interest.


In [ ]:
# Replace with actual record set @id from the listing above
if record_sets:
    main_record_set_id = record_sets[0]
else:
    main_record_set_id = None

# Load data from all discovered record sets
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records from record set '@id': {record_set_id}")
    recs = list(dataset.records(record_set=record_set_id))
    if len(recs) == 0:
        print(f"  No records found for '@id': {record_set_id}")
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")

# Show head for the main recordset
if main_record_set_id in dataframes:
    print(f"\nSample rows from '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print("No DataFrame loaded for main record set.")

## 4. Exploratory Data Analysis (EDA)

Apply some basic EDA operations: filtering on a numeric field, normalization, and grouping.

We'll select a numeric field `@id` (e.g., `_:peptide_count` or `_:normalized_abundance`) from the DataFrame columns above (replace with your actual column `@id`). For this demonstration, we pick the first numeric-looking field found.


In [ ]:
import numpy as np

df = dataframes.get(main_record_set_id)

# Identify a likely numeric field from DataFrame columns
numeric_candidate = None
if df is not None:
    for col in df.columns:
        # Try converting to numeric, if possible
        try:
            as_num = pd.to_numeric(df[col], errors='coerce')
            if as_num.notnull().sum() > df.shape[0] // 2:  # Choose column with numerics
                numeric_candidate = col
                break
        except Exception:
            continue

# For demo, fallback to user input:
if not numeric_candidate:
    numeric_candidate = df.columns[0] if df is not None and df.shape[1] > 0 else None
    print(f"No numeric field found; using first column: {numeric_candidate}")

print(f"Using field '@id': {numeric_candidate} for EDA.")

if numeric_candidate and df is not None:
    # Ensure it's numeric
    df[numeric_candidate] = pd.to_numeric(df[numeric_candidate], errors='coerce')
    # Filter for values > 10
    threshold = 10
    filtered_df = df[df[numeric_candidate] > threshold]
    print(f"Filtered rows with {numeric_candidate} > {threshold}: {len(filtered_df)} rows\n")
    display(filtered_df.head())
    # Normalize the field (z-score)
    filtered_df[f"{numeric_candidate}_normalized"] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
    print(f"Normalized {numeric_candidate} values:")
    display(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())
    # Try grouping by a text or categorical field
    group_field = None
    for col in df.columns:
        if col != numeric_candidate and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().reset_index()
        print(f"Grouped mean of {numeric_candidate} by {group_field} (top 5):")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No suitable DataFrame or numeric field for EDA.")

## 5. Visualization
We visualize the distribution of the selected numeric field and its normalized version.


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_candidate and df is not None:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    filtered_df[numeric_candidate].hist(bins=30)
    plt.title(f"Histogram: {numeric_candidate}")
    plt.xlabel(numeric_candidate)
    plt.ylabel('Count')
    
    if f"{numeric_candidate}_normalized" in filtered_df:
        plt.subplot(1, 2, 2)
        filtered_df[f"{numeric_candidate}_normalized"].hist(bins=30, color='orange')
        plt.title(f"Histogram: Normalized {numeric_candidate}")
        plt.xlabel(f"{numeric_candidate}_normalized")
        plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("No data to visualize.")

## 6. Conclusion

In this notebook, we:
- Loaded and explored the Croissant-structured dataset using `mlcroissant`, referencing all entities by their `@id` fields
- Presented all record sets and their field IDs
- Extracted and previewed records from all record sets
- Performed basic EDA: filtering, normalization, and grouping by key fields
- Visualized the distribution of a main numeric attribute

This template can be adapted for detailed analysis, model building, or integration with other FAIR data sources.
